# Reranker değerlendirmesi (Level 2)

**Ne yapar:** 20 golden query'yi dört modda ölçer — BM25, dense (BGE-M3),
hibrit (RRF) ve **rerank** (hibrit havuzunun cross-encoder ile yeniden
sıralanması). Çıktı `retrieval_eval.json`.

**Neden Colab:** yerel makinede GPU yok. `bge-reranker-v2-m3` (568M parametre)
CPU'da yalnızca **yüklenmesi** 574 saniye sürdü, ölçüm hiç bitmedi. T4'te
yükleme ~1 dakika, 20 sorunun tamamı birkaç dakika.

**Önce yapılacak:** `Runtime > Change runtime type > T4 GPU`.

**Tasarım notu:** Bu notebook değerlendirme kodunu **kopyalamıyor**. Yereldeki
`_scripts/retrieval_eval.py` olduğu gibi yükleniyor ve Colab'da aynı klasör
yapısı kurulup çalıştırılıyor. Kod kopyalansaydı iki sürüm zamanla ayrışır ve
hangi rakamın hangi mantıktan geldiği belirsizleşirdi.

In [ ]:
# 1) GPU var mı? Beklenen: 'Tesla T4' satırı ve cuda: True.
#    False görürsen Runtime menüsünden T4 seçmemişsindir; devam etme.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# 2) Kurulum. 1-2 dakika. Sonunda sürüm numarası yazar.
!pip install -q sentence-transformers
import sentence_transformers
print('sentence-transformers', sentence_transformers.__version__)

## Yüklenecek dosyalar

Aşağıdaki hücre dosya seçme penceresi açar. **Dokuz dosyayı birden seç**
(Ctrl ile çoklu seçim). Hepsi `C:\Users\Tunahan\Desktop\myeloma` altında:

| dosya | nerede | ne işe yarar |
|---|---|---|
| `chunks.json` | `_work\chunks\` | aranacak metinler |
| `golden_queries.json` | `_work\eval\` | 20 soru ve hedefleri |
| `BAAI_bge-m3.npy` | `_work\embeddings\` | hazır dense vektörler (~57 MB) |
| `BAAI_bge-m3.meta.json` | `_work\embeddings\` | vektörlerin künyesi |
| `retrieval_eval.py` | `_scripts\` | değerlendirici |
| `embed_chunks.py` | `_scripts\` | `gomulecek_metin` buradan geliyor |
| `query_processing.py` | `_scripts\` | sorgu işleme (7. adım) |
| `chunk.py` | `_scripts\` | `ILAC` sözlüğü buradan geliyor |
| `plog.py` | `_scripts\` | `embed_chunks`'ın bağımlılığı |

Vektörleri yeniden üretmiyoruz; Colab'da bir kez gömdüğümüz dosyayı geri
yüklüyoruz. Yeniden gömmek hem 37 dakika kaybettirir hem de sonuçların
önceki ölçümle karşılaştırılabilirliğini bozar.

In [ ]:
# 3) Yükleme + klasör yapısını kurma.
#    ÖNEMLİ: önce eski _scripts siliniyor. Bir önceki koşuda Colab oturumunda
#    kalan eski retrieval_eval.py çalıştı; sorgu işleme hiç devreye girmediği
#    halde sonuç geçerli göründü. Silmek bunu imkânsız kılıyor.
#    Dosyalar adlarına göre doğru klasöre dağıtılır; sırayla seçmen gerekmez.
#    Beklenen: dokuz satır 'yerlesti', ardından 'eksik: yok'.
import os, shutil
from google.colab import files

KOK = '/content/myeloma'
YER = {
    'chunks.json':           '_work/chunks',
    'golden_queries.json':   '_work/eval',
    'BAAI_bge-m3.npy':       '_work/embeddings',
    'BAAI_bge-m3.meta.json': '_work/embeddings',
    'retrieval_eval.py':     '_scripts',
    'embed_chunks.py':       '_scripts',
    'query_processing.py':   '_scripts',
    'chunk.py':              '_scripts',
    'plog.py':               '_scripts',
}
shutil.rmtree(os.path.join(KOK, '_scripts'), ignore_errors=True)
for d in set(YER.values()) | {'_logs'}:
    os.makedirs(os.path.join(KOK, d), exist_ok=True)

for ad in files.upload():
    # Colab ayni adli dosya varsa 'x (1).json' diye kaydediyor; adi geri al.
    temiz = ad.replace(' (1)', '').replace(' (2)', '')
    if temiz not in YER:
        print(f'  atlandi (tanimadim): {ad}')
        continue
    shutil.move(ad, os.path.join(KOK, YER[temiz], temiz))
    print(f'  yerlesti: {temiz} -> {YER[temiz]}')

eksik = [a for a, d in YER.items()
         if not os.path.exists(os.path.join(KOK, d, a))]
print('eksik:', eksik or 'yok')
assert not eksik, 'eksik dosya var - hücreyi tekrar çalıştırıp hepsini seç'

In [ ]:
# 4) Tutarlılık kontrolü. Üç şeyi doğruluyor, üçü de sessiz hata kaynağı.
import json, numpy as np

chunks = json.load(open(f'{KOK}/_work/chunks/chunks.json', encoding='utf-8'))
vek = np.load(f'{KOK}/_work/embeddings/BAAI_bge-m3.npy')
meta = json.load(open(f'{KOK}/_work/embeddings/BAAI_bge-m3.meta.json',
                      encoding='utf-8'))
print(f'chunk {len(chunks)}, vektör {vek.shape}, altkume_idx {meta["altkume_idx"]}')

# (a) Vektör sayısı chunk sayısına eşit olmalı. Değilse satır i'deki vektör
#     başka bir chunk'a ait olur ve skorlar anlamsızlaşır - hata vermeden.
assert meta['altkume_idx'] is None, 'bu dosya alt küme - tam arşiv gerekiyor'
assert vek.shape[0] == len(chunks), 'vektör/chunk sayısı uyuşmuyor'

# (b) Yüklenen script GÜNCEL mi? Bir önceki koşuda Colab eski sürümü
#     çalıştırdı ve sonuç, sorgu işleme hiç devreye girmediği halde geçerli
#     görünüyordu. Rakamlar makuldü, sadece yanlış konfigürasyona aitti.
kaynak = open(f'{KOK}/_scripts/retrieval_eval.py', encoding='utf-8').read()
assert '--qp' in kaynak, 'ESKİ retrieval_eval.py - 3. hücreyi tekrar çalıştır'
assert os.path.exists(f'{KOK}/_scripts/query_processing.py'), \
    'query_processing.py yüklenmemiş'

# (c) Sorgu işleme gerçekten ilaç tespit ediyor mu?
import sys
sys.path.insert(0, f'{KOK}/_scripts')
import query_processing as QP
sorular = json.load(open(f'{KOK}/_work/eval/golden_queries.json',
                         encoding='utf-8'))
bulan = sum(1 for s in sorular if QP.analiz(s['soru'])['ilac'])
print(f'sorgu işleme: {bulan}/{len(sorular)} soruda ilaç tespit edildi '
      f'(yerelde 10 bekleniyor)')
assert bulan >= 8, 'ilaç tespiti beklenenden düşük - sözlük eksik yüklenmiş'
print('tamam')

In [ ]:
# 5) Asıl ölçüm: sorgu işleme AÇIK, dört mod.
#    --qp = boru hattının 7. adımı: sorudaki etken maddeyi tanı, BM25 sorgusunu
#    İngilizce INN ve ATC ile genişlet, aday listesini ilaç uyumuna göre yeniden
#    diz. Yerelde ölçüldü, GPU'suz üç modda da kazandırdı (örn. hibrit MRR
#    0,569 -> 0,679). Kalan soru: reranker'ın üzerine ne koyduğu.
#    HF_HOME'u burada veriyoruz: script'teki varsayılan D:/hf_cache, Colab'da yok.
%env HF_HOME=/content/hf_cache
!cd {KOK} && python _scripts/retrieval_eval.py --mod hepsi --qp --detay

In [ ]:
# 6) İki karşılaştırma. Toplam birkaç dakika.
#    (a) Sorgu işleme reranker'a da yarıyor mu? Aynı mod, --qp'siz.
#    (b) Havuz büyüklüğü: reranker ancak havuza giren adayı yukarı çekebilir.
#        Havuzu büyütmek recall tavanını yükseltir, maliyeti de artırır.
#        Havuz 100 daha önce soru başına 9,4 sn, havuz 50 ise 4,2 sn sürmüştü.
!cd {KOK} && python _scripts/retrieval_eval.py --mod rerank
!cd {KOK} && python _scripts/retrieval_eval.py --mod rerank --qp --havuz 20
!cd {KOK} && python _scripts/retrieval_eval.py --mod rerank --qp --havuz 100

In [ ]:
# 7) Raporu indir. NOT: 6. hücreyi çalıştırdıysan rapor son koşuya aittir;
#    dört modun tablosu için önce 5. hücreyi tekrar çalıştır, sonra indir.
from google.colab import files
files.download(f'{KOK}/_logs/retrieval_eval.json')

## Bittikten sonra

İnen `retrieval_eval.json` dosyasını şuraya koy:

```
C:\Users\Tunahan\Desktop\myeloma\_logs\
```

Ekrandaki tabloları da bana yapıştır — indirilen dosya yalnızca **son** koşuya
ait, oysa karşılaştırma için hepsi gerekiyor. Rapora artık `havuz` ve `qp`
alanları da yazılıyor, yani hangi ayarla ölçüldüğü dosyadan okunabiliyor.

Karşılaştırma tabanı (sorgu işleme kapalı, havuz 50):

| mod | recall@5 | recall@10 | MRR | kaçan |
|---|---|---|---|---|
| BM25 | 0,60 | 0,70 | 0,469 | 6 |
| dense | 0,85 | 0,85 | 0,660 | 3 |
| hibrit | 0,70 | 0,90 | 0,569 | 2 |
| rerank | 0,75 | 0,90 | 0,748 | 2 |

Sorgu işleme açıkken yerelde ölçülenler (reranker hariç):

| mod | recall@5 | recall@10 | MRR | kaçan |
|---|---|---|---|---|
| BM25 | 0,65 | 0,70 | 0,618 | 6 |
| dense | 0,90 | 0,90 | 0,713 | 2 |
| hibrit | 0,75 | 0,95 | 0,679 | 1 |

Bakacağım şey: reranker + sorgu işleme birlikte MRR'ı 0,748'in üzerine
çıkarıyor mu, yoksa ikisi aynı hataları mı düzeltiyor? İkincisi ise reranker'ın
maliyetini ödemeye değmeyebilir, çünkü sorgu işleme neredeyse bedava.